In [ ]:
# This is a Jupyter notebook. Create a new Jupyter notebook and paste this content.

"""
# Exploratory Analysis of Emotional Manipulation in LLMs

This notebook performs initial exploratory analysis of emotional manipulation patterns in LLM responses.
"""

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# 1. Load Data
print("Loading data...")

# Load attack prompts
with open('../data/raw/attack_prompts.json', 'r') as f:
    attack_prompts = json.load(f)

# Load baseline prompts
with open('../data/raw/baseline_prompts.json', 'r') as f:
    baseline_prompts = json.load(f)

print(f"Loaded {len(attack_prompts)} attack prompts")
print(f"Loaded {len(baseline_prompts)} baseline prompts")

# 2. Data Exploration
print("\n=== Data Exploration ===")

# Create DataFrames
attack_df = pd.DataFrame(attack_prompts)
baseline_df = pd.DataFrame(baseline_prompts)

# Display sample
print("Sample attack prompts:")
print(attack_df[['id', 'prompt', 'metadata']].head())

# 3. Analyze Manipulation Types
print("\n=== Manipulation Type Analysis ===")

# Extract manipulation types
attack_df['manipulation_type'] = attack_df['metadata'].apply(lambda x: x['manipulation_type'])
attack_df['manipulation_level'] = attack_df['metadata'].apply(lambda x: x['manipulation_level'])
attack_df['target_task'] = attack_df['metadata'].apply(lambda x: x['target_task'])

# Count by manipulation type
type_counts = attack_df['manipulation_type'].value_counts()
level_counts = attack_df['manipulation_level'].value_counts()
task_counts = attack_df['target_task'].value_counts()

print("Manipulation Type Distribution:")
print(type_counts)
print("\nManipulation Level Distribution:")
print(level_counts)
print("\nTarget Task Distribution:")
print(task_counts)

# 4. Visualization
print("\n=== Visualizations ===")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Manipulation Type Distribution
axes[0,0].bar(type_counts.index, type_counts.values)
axes[0,0].set_title('Distribution of Manipulation Types', fontsize=14)
axes[0,0].set_ylabel('Count', fontsize=12)
axes[0,0].tick_params(axis='x', rotation=45)

# Plot 2: Manipulation Level Distribution
axes[0,1].bar(level_counts.index, level_counts.values)
axes[0,1].set_title('Distribution of Manipulation Levels', fontsize=14)
axes[0,1].set_ylabel('Count', fontsize=12)

# Plot 3: Target Task Distribution
axes[1,0].bar(task_counts.index, task_counts.values)
axes[1,0].set_title('Distribution of Target Tasks', fontsize=14)
axes[1,0].set_ylabel('Count', fontsize=12)
axes[1,0].tick_params(axis='x', rotation=45)

# Plot 4: Combined Distribution
# Create cross-tabulation
cross_tab = pd.crosstab(attack_df['manipulation_type'], attack_df['manipulation_level'])
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,1])
axes[1,1].set_title('Manipulation Type × Level Heatmap', fontsize=14)

plt.tight_layout()
plt.savefig('../data/results/exploratory_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Text Analysis
print("\n=== Text Analysis ===")

# Calculate basic text statistics
attack_df['prompt_length'] = attack_df['prompt'].apply(len)
attack_df['word_count'] = attack_df['prompt'].apply(lambda x: len(str(x).split()))

baseline_df['prompt_length'] = baseline_df['prompt'].apply(len)
baseline_df['word_count'] = baseline_df['prompt'].apply(lambda x: len(str(x).split()))

print("Attack Prompts Text Statistics:")
print(f"Average length: {attack_df['prompt_length'].mean():.1f} characters")
print(f"Average word count: {attack_df['word_count'].mean():.1f} words")
print(f"Min length: {attack_df['prompt_length'].min()} characters")
print(f"Max length: {attack_df['prompt_length'].max()} characters")

print("\nBaseline Prompts Text Statistics:")
print(f"Average length: {baseline_df['prompt_length'].mean():.1f} characters")
print(f"Average word count: {baseline_df['word_count'].mean():.1f} words")
print(f"Min length: {baseline_df['prompt_length'].min()} characters")
print(f"Max length: {baseline_df['prompt_length'].max()} characters")

# 6. Sentiment Analysis (Optional - requires nltk or textblob)
try:
    from textblob import TextBlob
    
    print("\n=== Sentiment Analysis ===")
    
    attack_df['sentiment'] = attack_df['prompt'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
    baseline_df['sentiment'] = baseline_df['prompt'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
    
    print(f"Attack prompts average sentiment: {attack_df['sentiment'].mean():.3f}")
    print(f"Baseline prompts average sentiment: {baseline_df['sentiment'].mean():.3f}")
    
    # Plot sentiment distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].hist(attack_df['sentiment'], bins=20, alpha=0.7, color='red')
    axes[0].set_title('Attack Prompts Sentiment Distribution', fontsize=14)
    axes[0].set_xlabel('Sentiment Polarity', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    
    axes[1].hist(baseline_df['sentiment'], bins=20, alpha=0.7, color='green')
    axes[1].set_title('Baseline Prompts Sentiment Distribution', fontsize=14)
    axes[1].set_xlabel('Sentiment Polarity', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('../data/results/sentiment_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
except ImportError:
    print("TextBlob not installed. Install with: pip install textblob")

# 7. Save Analysis Results
print("\n=== Saving Analysis Results ===")

analysis_results = {
    'dataset_summary': {
        'attack_prompts_count': len(attack_prompts),
        'baseline_prompts_count': len(baseline_prompts),
        'manipulation_types': type_counts.to_dict(),
        'manipulation_levels': level_counts.to_dict(),
        'target_tasks': task_counts.to_dict()
    },
    'text_statistics': {
        'attack_prompts': {
            'avg_length': float(attack_df['prompt_length'].mean()),
            'avg_word_count': float(attack_df['word_count'].mean()),
            'min_length': int(attack_df['prompt_length'].min()),
            'max_length': int(attack_df['prompt_length'].max())
        },
        'baseline_prompts': {
            'avg_length': float(baseline_df['prompt_length'].mean()),
            'avg_word_count': float(baseline_df['word_count'].mean()),
            'min_length': int(baseline_df['prompt_length'].min()),
            'max_length': int(baseline_df['prompt_length'].max())
        }
    }
}

# Save to JSON
with open('../data/results/exploratory_analysis.json', 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f"Analysis results saved to ../data/results/exploratory_analysis.json")

# 8. Generate Report
print("\n=== Analysis Complete ===")
print("\nKey Findings:")
print(f"1. Total attack prompts: {len(attack_prompts)}")
print(f"2. Most common manipulation type: {type_counts.idxmax()} ({type_counts.max()} instances)")
print(f"3. Most common target task: {task_counts.idxmax()} ({task_counts.max()} instances)")
print(f"4. Average attack prompt length: {attack_df['prompt_length'].mean():.1f} characters")
print(f"5. Average baseline prompt length: {baseline_df['prompt_length'].mean():.1f} characters")

# 01 Exploratory Analysis

This notebook explores the attack prompts and baseline prompts.

## Load data

```python
import json

with open('../data/raw/attack_prompts.json', 'r') as f:
    attack_prompts = json.load(f)

with open('../data/raw/baseline_prompts.json', 'r') as f:
    baseline_prompts = json.load(f)